# Script to get the differences of emissions between scenarios

This jupyter notebook contains all routine calculations of the emission differences in Europe compared to another scenario. 

**Authors:** Johannes Giehl (jfg.eco@cbs.dk)

## Import packages

In [1]:
#import
import pandas as pd
from cost_and_gas_source_share_functions import *

In [84]:
def filter_negative_supply(df, commodity_name):
    """
    Filters the DataFrame to a given commodity and removes rows with Supply >= 0.
    Handles column name capitalization issues.
    """
    # Standardize column names (optional safety step)
    df = df.rename(columns={col: col.strip().capitalize() for col in df.columns})
    
    # Check for required columns
    required_columns = {'Commodity', 'Node', 'Supply'}
    if not required_columns.issubset(df.columns):
        raise ValueError(f"Missing required columns. Found columns: {df.columns.tolist()}")

    # Filter and return
    df_filtered = df[(df['Commodity'] == commodity_name) & (df['Supply'] < 0)].copy()

    # make values now positive for later emissions calculations
    df_filtered['Supply'] = df_filtered['Supply'] * -1  
    
    return df_filtered.reset_index(drop=True)

def calculate_emissions_per_country(emission_factors, supply_shares, demand):
    # Extract country/region code before the first underscore (supports 2- or 3-letter codes)
    supply_shares['Source'] = supply_shares['Origin'].str.extract(r'^([A-Z]+)_')

    # Merge with demand
    df = supply_shares.merge(demand, on='Node')

    # Safe lookup for emission factors with fallback to 0
    def get_factor(node, source):
        try:
            return emission_factors.at[node, source]
        except KeyError:
            return 0

    # Calculate emission factor and emissions
    df['EmissionFactor'] = df.apply(lambda row: get_factor(row['Node'], row['Source']), axis=1)
    df['Emissions'] = df['Supply'] * df['Share'] * df['EmissionFactor']

    # Aggregate emissions per demand country
    return df.groupby('Node')['Emissions'].sum().reset_index()

def calculate_domestic_emissions(supply_shares, supply_df, dfs_emission_factors_dom):
    """
    Calculates emissions from domestic production only.
    
    Parameters:
    - supply_shares: DataFrame with 'Node', 'Origin', 'Share'
    - supply_df: DataFrame with 'Node', 'Supply' (positive values)
    - dfs_emission_factors_dom: DataFrame with index = 'Node', column = 'Factor'
    
    Returns:
    - DataFrame with 'Node' and 'Domestic_Emissions'
    """
    # Extract Source country code from Origin
    supply_shares['Source'] = supply_shares['Origin'].str.extract(r'^([A-Z]+)_')

    # Keep only domestic rows (where Node == Source)
    domestic = supply_shares[supply_shares['Node'] == supply_shares['Source']].copy()

    # Merge with total supply to compute actual domestic amount
    df = domestic.merge(supply_df, on='Node')  # Now we have 'Share' and total 'Supply'

    # Calculate domestic supply = total supply × share
    df['Domestic_Supply'] = df['Supply'] * df['Share']

    # Add domestic emission factor from dfs_emission_factors_dom
    df = df.merge(dfs_emission_factors_dom, on='Node')  # Adds 'Factor'

    # Emissions = domestic supply × domestic factor
    df['Domestic_Emissions'] = df['Domestic_Supply'] * df['Factor']

    # Return result
    return df[['Node', 'Domestic_Emissions']].reset_index(drop=True)

def combine_total_emissions(import_emissions_df, domestic_emissions_df):
    """
    Combines import and domestic emissions into a single DataFrame with totals,
    and removes rows with zero total emissions.
    
    Parameters:
    - import_emissions_df: DataFrame with ['Node', 'Emissions']
    - domestic_emissions_df: DataFrame with ['Node', 'Domestic_Emissions']
    
    Returns:
    - DataFrame with ['Node', 'Import_Emissions', 'Domestic_Emissions', 'Total_Emissions']
      excluding rows where total emissions are zero.
    """
    # Rename for clarity
    import_emissions = import_emissions_df.rename(columns={'Emissions': 'Import_Emissions'})
    domestic_emissions = domestic_emissions_df.rename(columns={'Domestic_Emissions': 'Domestic_Emissions'})
    
    # Merge both DataFrames on Node, fill missing with 0
    combined = pd.merge(import_emissions, domestic_emissions, on='Node', how='outer').fillna(0)

    # Add total emissions column
    combined['Total_Emissions'] = combined['Import_Emissions'] + combined['Domestic_Emissions']

    # Filter out rows with 0 total emissions
    combined = combined[combined['Total_Emissions'] > 0].reset_index(drop=True)

    return combined[['Node', 'Import_Emissions', 'Domestic_Emissions', 'Total_Emissions']]

def calculate_total_emissions(combined_emissions_df):
    """
    Calculates total import, domestic, and combined emissions from the combined DataFrame.
    
    Parameters:
    - combined_emissions_df: DataFrame with columns:
        - 'Import_Emissions'
        - 'Domestic_Emissions'
        - 'Total_Emissions'
    
    Returns:
    - Dictionary with total values for each emission type
    """
    totals = {
        'Total_Import_Emissions': combined_emissions_df['Import_Emissions'].sum(),
        'Total_Domestic_Emissions': combined_emissions_df['Domestic_Emissions'].sum(),
        'Total_Emissions': combined_emissions_df['Total_Emissions'].sum()
    }
    return totals

def build_emissions_summary(**kwargs):
    """
    Combines multiple emissions totals dicts into a single DataFrame.
    
    Parameters:
    - kwargs: named totals dictionaries, e.g., totals_2021=..., totals_2024=...
    
    Returns:
    - DataFrame with scenario names as rows and emissions metrics as columns
    """
    records = []
    for key, totals_dict in kwargs.items():
        # Extract label after 'totals_' for row name
        label = key.replace('totals_', '')
        row = {'Scenario': label}
        row.update(totals_dict)
        records.append(row)
    
    return pd.DataFrame(records).set_index('Scenario')

## Define path to data

In [46]:
#path for output and input data
#for gas shares per country
data_file_path = os.path.join('..', '..', '01_data', '02_output_data', '02_unidirectional_results', '01_paper_IAEE', '02_prepared_results')
full_data_file_path = os.path.abspath(os.path.join(os.getcwd(), data_file_path))
#for demand per country
demand_data_path = os.path.join('..', '..', '01_data', '01_input_data', '02_processed', '01_paper_IAEE')
full_demand_data_path = os.path.abspath(os.path.join(os.getcwd(), demand_data_path))
#emission factors
emission_factors_data_path = os.path.join('..', '..', '01_data', '01_input_data', '02_processed')
emission_factors_file_name = '\emission_factors_natural_gas.xlsx'
emission_factors_path = emission_factors_data_path + emission_factors_file_name
full_emission_factors_path = os.path.abspath(os.path.join(os.getcwd(), emission_factors_path))

#specify the output file
output_file_name = '\emission_differences.xlsx'
output_file_path = data_file_path + output_file_name
full_output_file_path = os.path.abspath(os.path.join(os.getcwd(), output_file_path))

## Load te data

In [66]:
#get all cost dfs from the excels in the prepared results folder
#shares of gas sources per node
dfs = load_excel_sheets_by_name(full_data_file_path, "costs_shares", sheet_name="shares")
#demand of the nodes
dfs_demand = load_excel_sheets_by_name(full_demand_data_path, "inputs_IAEE_2025_run", sheet_name="Supply")
#emission factors imports
dfs_emission_factors_im = pd.read_excel(full_emission_factors_path, sheet_name="Emission_Factors_Import", index_col=0)
#emission factors domestic
dfs_emission_factors_dom = pd.read_excel(full_emission_factors_path, sheet_name="Emission_Factors_Domestic", index_col=0)

In [67]:
dfs_emission_factors_dom

,Factor
Node,
AL,3
AT,3
BE,3
BA,3
BG,3
HR,3
CZ,3
DK,3
EE,3


## Prepare individual dataframes

In [12]:
df_names = list(dfs.keys())
df_names

['costs_shares_IAEE_2025_run_2021',
 'costs_shares_IAEE_2025_run_2024',
 'costs_shares_IAEE_2025_run_2024_inv',
 'costs_shares_IAEE_2025_run_2024_plus_no_QA',
 'costs_shares_IAEE_2025_run_2024_plus_NO_reduced',
 'costs_shares_IAEE_2025_run_2024_plus_no_USA',
 'costs_shares_IAEE_2025_run_2024_with_RU',
 'costs_shares_IAEE_2025_run_2035_AP',
 'costs_shares_IAEE_2025_run_2035_SP']

In [13]:
df_shares_IAEE_2025_run_2021 = dfs['costs_shares_IAEE_2025_run_2021']
df_shares_IAEE_2025_run_2024 = dfs['costs_shares_IAEE_2025_run_2024']
df_shares_IAEE_2025_run_2024_inv = dfs['costs_shares_IAEE_2025_run_2024_inv']
df_shares_IAEE_2025_run_2024_plus_no_QA = dfs['costs_shares_IAEE_2025_run_2024_plus_no_QA']
df_shares_IAEE_2025_run_2024_plus_NO_reduced = dfs['costs_shares_IAEE_2025_run_2024_plus_NO_reduced']
df_shares_IAEE_2025_run_2024_plus_no_USA = dfs['costs_shares_IAEE_2025_run_2024_plus_no_USA']
df_shares_IAEE_2025_run_2024_with_RU = dfs['costs_shares_IAEE_2025_run_2024_with_RU']
df_shares_IAEE_2025_run_2035_AP = dfs['costs_shares_IAEE_2025_run_2035_AP']
df_shares_IAEE_2025_run_2035_SP = dfs['costs_shares_IAEE_2025_run_2035_SP']

In [16]:
df_names_demand = list(dfs_demand.keys())
df_names_demand

['inputs_IAEE_2025_run_2021',
 'inputs_IAEE_2025_run_2024',
 'inputs_IAEE_2025_run_2024_inv',
 'inputs_IAEE_2025_run_2024_plus_no_QA',
 'inputs_IAEE_2025_run_2024_plus_NO_reduced',
 'inputs_IAEE_2025_run_2024_plus_no_USA',
 'inputs_IAEE_2025_run_2024_with_RU',
 'inputs_IAEE_2025_run_2035_AP',
 'inputs_IAEE_2025_run_2035_SP']

In [19]:
df_demand_IAEE_2025_run_2021                 = dfs_demand['inputs_IAEE_2025_run_2021']
df_demand_IAEE_2025_run_2024                 = dfs_demand['inputs_IAEE_2025_run_2024']
df_demand_IAEE_2025_run_2024_inv             = dfs_demand['inputs_IAEE_2025_run_2024_inv']
df_demand_IAEE_2025_run_2024_plus_no_QA      = dfs_demand['inputs_IAEE_2025_run_2024_plus_no_QA']
df_demand_IAEE_2025_run_2024_plus_NO_reduced = dfs_demand['inputs_IAEE_2025_run_2024_plus_NO_reduced']
df_demand_IAEE_2025_run_2024_plus_no_USA     = dfs_demand['inputs_IAEE_2025_run_2024_plus_no_USA']
df_demand_IAEE_2025_run_2024_with_RU         = dfs_demand['inputs_IAEE_2025_run_2024_with_RU']
df_demand_IAEE_2025_run_2035_AP              = dfs_demand['inputs_IAEE_2025_run_2035_AP']
df_demand_IAEE_2025_run_2035_SP              = dfs_demand['inputs_IAEE_2025_run_2035_SP']

In [36]:
#filter the dataframes so that they contain only demand
df_demand_2021_filtered = filter_negative_supply(df_demand_IAEE_2025_run_2021, "Methane")
df_demand_2024_filtered = filter_negative_supply(df_demand_IAEE_2025_run_2024, "Methane")
df_demand_2024_inv_filtered = filter_negative_supply(df_demand_IAEE_2025_run_2024_inv, "Methane")
df_demand_2024_no_QA_filtered = filter_negative_supply(df_demand_IAEE_2025_run_2024_plus_no_QA, "Methane")
df_demand_2024_NO_red_filtered = filter_negative_supply(df_demand_IAEE_2025_run_2024_plus_NO_reduced, "Methane")
df_demand_2024_no_USA_filtered = filter_negative_supply(df_demand_IAEE_2025_run_2024_plus_no_USA, "Methane")
df_demand_2024_with_RU_filtered = filter_negative_supply(df_demand_IAEE_2025_run_2024_with_RU, "Methane")
df_demand_2035_AP_filtered = filter_negative_supply(df_demand_IAEE_2025_run_2035_AP, "Methane")
df_demand_2035_SP_filtered = filter_negative_supply(df_demand_IAEE_2025_run_2035_SP, "Methane")

## Calculate emissions

In [72]:
df_import_emissions_2021 = calculate_emissions_per_country(dfs_emission_factors_im, df_shares_IAEE_2025_run_2021, df_demand_2021_filtered)

In [73]:
df_domestic_emissions_2021 = calculate_domestic_emissions(df_shares_IAEE_2025_run_2021, df_demand_2021_filtered, dfs_emission_factors_dom)

In [77]:
df_total_emissions_2021 = combine_total_emissions(df_import_emissions_2021, df_domestic_emissions_2021)

In [83]:
totals_2021 = calculate_total_emissions(df_total_emissions_2021)

In [82]:
totals

{'Total_Import_Emissions': 89131353.42560573,
 'Total_Domestic_Emissions': 2576184.0541029926,
 'Total_Emissions': 91707537.47970873}

In [85]:
df_emissions_summary = build_emissions_summary(
    totals_2021=totals_2021,
    totals_2024=totals_2021,
    totals_2024_inv=totals_2021
)

In [86]:
df_emissions_summary

,Total_Import_Emissions,Total_Domestic_Emissions,Total_Emissions
Scenario,,,
2021,8.913135e+07,2.576184e+06,9.170754e+07
2024,8.913135e+07,2.576184e+06,9.170754e+07
2024_inv,8.913135e+07,2.576184e+06,9.170754e+07
